In [ ]:
import sys
from pathlib import Path

current_dir = Path.cwd().resolve()

# Find 'src' by walking up the directory tree
src_path = None
for parent in [current_dir] + list(current_dir.parents):
    candidate = parent / 'src'
    if candidate.exists() and candidate.is_dir():
        src_path = candidate
        break

if src_path:
    if str(src_path) not in sys.path:
        sys.path.append(str(src_path))
    print(f"Success: Linked to src at {src_path}")
    
    # Import
    try:
        from config import *
        print(f"Environment linked. Data dir: {DATA_DIR}")    
    except ImportError as e:
        print(f"CRITICAL: Found src but could not import config. {e}")
else:
    print("CRITICAL ERROR: Could not find 'src' directory in any parent folder.")
    print(f"Current search path: {current_dir}")

In [ ]:
from tqdm import tqdm
import numpy as np
import pandas as pd
from mixing_utils import mix_signals

TARGET_LEN = 40960
SINR_RANGE = (-10, 20)

In [ ]:
def generate_evaluation_set(split_name):
    print(f"\n=== Generating {split_name.upper()} Set (1:1 Random Mix) ===")
    
    # Load Raw Split Data
    clean_path = get_unet_path(STAGE_AUGMENTED, split=split_name, signal=CommSignal2)
    clean_signals_raw = np.load(clean_path)

    barrage_path = get_unet_path(STAGE_SPLIT, split=split_name, signal=BarrageSignal)
    barrage_data = np.load(barrage_path)
    
    # Verify Counts
    n_clean = len(clean_signals_raw)
    n_noise = len(barrage_data)

    if n_clean != n_noise:
        print(f"WARNING: Count mismatch! Clean: {n_clean}, Barrage: {n_noise}")
        limit = min(n_clean, n_noise)
    else:
        limit = n_clean
        print(f"Verified 1:1 match: {limit} samples.")
    
    # Pre-calculate Random SINRs
    # We sample uniformly from -10dB to +20dB to test general performance
    target_sinrs = np.random.uniform(SINR_RANGE[0], SINR_RANGE[1], size=limit)
    
    mixed_signals = []
    metadata_rows = []
    
    print(f"Mixing {limit} samples...")

    for i in tqdm(range(limit)):
        raw_sig = clean_signals_raw[i]
        raw_noise = barrage_data[i]
        sinr_db = target_sinrs[i]

        # --- Validation & Cropping ---
        if raw_sig.shape[1] < TARGET_LEN:
            print(f"Skip: Clean sample {i} too short.")
            continue
        if raw_noise.shape[1] < TARGET_LEN:
            print(f"Skip: Noise sample {i} too short.")
            continue

        # Deterministic Crop (Start of file)
        sig_clean = raw_sig[:, :TARGET_LEN]
        sig_noise = raw_noise[:, :TARGET_LEN]

        # --- Mixing ---
        sig_mixed, alpha = mix_signals(sig_clean, sig_noise, sinr_db)
        
        # Store
        mixed_signals.append(sig_mixed)
        metadata_rows.append({
            'global_index': len(mixed_signals) - 1, # Sequential index
            'sinr_db': round(sinr_db, 4),
            'clean_frame_id': i,
            'noise_frame_id': i,
            'interference_type': 'Barrage',
            'source': 'AWGN_Generator',
            'alpha': round(alpha, 6)
        })
            
    # Save to Disk
    output_dir = get_unet_path(STAGE_MIXED, split=split_name)
    output_npy = output_dir / f"{MIXED_DATASET}.npy"
    output_csv = output_dir / f"{MIXED_METADATA}.csv"
    
    # Save mixed tensor
    np.save(output_npy, np.array(mixed_signals))
    
    # Save metadata
    df = pd.DataFrame(metadata_rows)
    df.to_csv(output_csv, index=False)
    
    print(f"Success! Saved {len(mixed_signals)} samples to {output_npy}")
    print(f"Metadata saved to {output_csv}")

In [ ]:
generate_evaluation_set(VAL)
generate_evaluation_set(TEST)